In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score


In [ ]:
df = pd.read_csv("dataset.csv")

# We define the EXACT column names expected in the CSV
feature_cols = ["income", "expenses", "loanamount", "term_years"]
target_col = "loan_approved"

# Verify all necessary columns exist in the dataset
missing = [c for c in feature_cols + [target_col] if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns in dataset.csv: {missing}")

# Separate the features (the questions) from the target (the answer key)
X, y = df[feature_cols].copy(), df[target_col].copy()

# Convert text-based targets (like Yes/No) to 1/0 for the AI
if not np.issubdtype(y.dtype, np.number):
    y, mapping = pd.factorize(y)



In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")), # Fills in blank boxes
    ("scaler", StandardScaler()),                  # Shrinks numbers to a fair scale
])

preprocess = ColumnTransformer([("num", numeric_pipeline, feature_cols)])

clf = Pipeline([
    ("preprocess", preprocess),
    ("model", LogisticRegression(max_iter=2000, class_weight="balanced")) # The Math Brain
])

In [ ]:
clf.fit(X_train, y_train) # AI learns from the training data

# Grade the AI on both the study guide and the final exam
train_acc = accuracy_score(y_train, clf.predict(X_train))
test_acc = accuracy_score(y_test, clf.predict(X_test))
gap = train_acc - test_acc

print(f"Training Acc: {train_acc:.4f} | Testing Acc: {test_acc:.4f} | Gap: {gap:.4f}")
if gap > 0.05:
    print("Warning: Overfitting! (The AI just memorized the training data).")


Training Acc: 0.8936 | Testing Acc: 0.8868 | Gap: 0.0068


In [ ]:
print("\n--- Loan Predictor (Type Ctrl+C to stop) ---")
while True:
    try:
        # Collect info and package it directly into a DataFrame
        user_df = pd.DataFrame([{
            "income": float(input("Yearly income: ")),
            "expenses": float(input("Yearly expenses: ")),
            "loanamount": float(input("Loan amount: ")),
            "term_years": float(input("Loan term (years): "))
        }])

        # Get AI's decision
        pred = clf.predict(user_df)[0]
        prob = clf.predict_proba(user_df)[0][1]

        print(f">> Prediction: {'APPROVED' if pred == 1 else 'REJECTED'}")
        print(f">> Confidence: {prob:.2%} chance of approval\n")

        if input("Check another? (y/n): ").strip().lower() != "y":
            break

    except KeyboardInterrupt:
        break
    except ValueError:
        print("[!] Error: Please type valid numbers.\n")


--- Loan Predictor (Type Ctrl+C to stop) ---
Yearly income: 150000
Yearly expenses: 55000
Loan amount: 250000
Loan term (years): 10
>> Prediction: APPROVED
>> Confidence: 70.89% chance of approval

Check another? (y/n): n
